# 07 — Seniority from Description (Experience-Based)

**Motivation:** the existing `seniority` column (title-keyword based) buckets 57% of postings (2,843 of 4,949) into `Mid/Unspecified` — a catch-all that conflates true mid-level roles with postings where the title just didn't signal a tier. That ambiguity is flagged as an open caveat in `Inferences.md`.

This notebook extracts required years-of-experience mentions from the `description` column via regex, maps them to a seniority tier independent of title, and compares the two signals — with the goal of resolving `Mid/Unspecified` rows where the description actually states a clear experience requirement, not replacing the title-based column outright.

**Coverage note (tested before writing this notebook):** the experience-mention regex matches ~33% of the full dataset, and resolves 28.6% of the `Mid/Unspecified` rows specifically (812 of 2,843). This is a *supplementary* signal, not a full replacement — most postings simply don't state years of experience explicitly, and this notebook doesn't invent a number where none exists.

In [1]:
import pandas as pd
import re

IN_PATH = "../Outputs/mia_postings_final2_fixed2.csv"
OUT_PATH = "../Outputs/mia_postings_seniority_enriched.csv"

df = pd.read_csv(IN_PATH)
print(df.shape)
df["seniority"].value_counts()

(4949, 18)


seniority
Mid/Unspecified       2843
Manager/Lead           967
Senior                 672
Entry/Junior           248
Director/Executive     219
Name: count, dtype: int64

## 1. Extraction patterns

Three patterns, all anchored to the word "experience" (within the match itself) to avoid false positives like "10 years in business" or "25 years of history" that mention *years* but not *required experience*:

- `EXP_RANGE` — "3-5 years of experience", "2 to 4 years experience"
- `EXP_MIN` — "minimum of 3 years experience", "at least 5 years' experience"
- `EXP_PLUS` — "5+ years of experience"

Each description can match multiple times (requirements sections often restate the number). All matches are kept and reconciled per-posting in step 2.

In [2]:
EXP_RANGE = re.compile(
    r"(\d{1,2})\s*(?:\+|\-|to|\u2013|\u2014)\s*(\d{1,2})?\s*\+?\s*years?"
    r"(?:\s*\(?[a-z\s]{0,15}\)?)?\s*(?:of\s+)?experience",
    re.IGNORECASE,
)
EXP_MIN = re.compile(
    r"(?:minimum|at least|min\.?)\s*(?:of\s*)?(\d{1,2})\s*\+?\s*years?\s*(?:of\s+)?experience",
    re.IGNORECASE,
)
EXP_PLUS = re.compile(
    r"(\d{1,2})\s*\+\s*years?\s*(?:of\s+)?experience",
    re.IGNORECASE,
)


def extract_years(desc):
    """Return list of (low, high) year tuples found in a description, or None."""
    if not isinstance(desc, str):
        return None
    years = []
    for m in EXP_RANGE.finditer(desc):
        lo = int(m.group(1))
        hi = int(m.group(2)) if m.group(2) else lo
        if lo > hi:
            lo, hi = hi, lo
        years.append((lo, hi))
    for m in EXP_MIN.finditer(desc):
        v = int(m.group(1))
        years.append((v, v))
    for m in EXP_PLUS.finditer(desc):
        v = int(m.group(1))
        years.append((v, v))
    # sanity filter: drop implausible values (data-entry/typo noise, e.g. "100 years experience")
    years = [(lo, hi) for lo, hi in years if 0 <= lo <= 25 and 0 <= hi <= 25]
    return years if years else None


df["years_found"] = df["description"].apply(extract_years)
coverage = df["years_found"].notna().mean()
print(f"Coverage: {coverage:.1%} of postings have at least one experience-year mention")

Coverage: 33.3% of postings have at least one experience-year mention


## 2. Reconcile multiple mentions per posting

A posting can mention experience requirements more than once (e.g. once in a summary, again in a bulleted requirements list). Take the **median of the low-end values** found — more robust to a single outlier mention than min, and doesn't silently pick the max (which would systematically push borderline postings into a more senior bucket than intended).

In [3]:
import statistics


def reconcile(years_list):
    if not years_list:
        return None
    lows = [lo for lo, hi in years_list]
    return statistics.median(lows)


df["experience_years_required"] = df["years_found"].apply(reconcile)
df["experience_years_required"].describe()

count    1648.000000
mean        3.940837
std         2.569671
min         0.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        16.000000
Name: experience_years_required, dtype: float64

## 3. Map years to a seniority tier

Thresholds below are a **stated heuristic, not an industry standard** — document them as such in the report rather than presenting the mapping as authoritative:

| Years required | Tier |
|---|---|
| 0–1 | Entry/Junior |
| 2–4 | Mid (from experience) |
| 5–7 | Senior |
| 8–9 | Manager/Lead |
| 10+ | Director/Executive |

This uses the **same tier labels** as the existing title-based `seniority` column so the two are directly comparable, with one addition (`Mid (from experience)`) to distinguish a description-confirmed mid-level posting from the original catch-all `Mid/Unspecified`.

In [4]:
def years_to_tier(years):
    # NB: pandas silently converts None to NaN (a float) once a column has any numeric
    # values, so `years is None` never fires here — confirmed by testing this against the
    # real data before writing it this way. The `is None` version silently mislabeled every
    # unmatched row as Director/Executive (the final fallback branch below), because
    # `nan <= 1`, `nan <= 4`, etc. all evaluate to False in Python, so execution fell through
    # every comparison to the last `return` statement. Use pd.isna(), not `is None`.
    if pd.isna(years):
        return None
    if years <= 1:
        return "Entry/Junior"
    if years <= 4:
        return "Mid (from experience)"
    if years <= 7:
        return "Senior"
    if years <= 9:
        return "Manager/Lead"
    return "Director/Executive"


df["seniority_from_experience"] = df["experience_years_required"].apply(years_to_tier)
df["seniority_from_experience"].value_counts(dropna=False)

seniority_from_experience
NaN                      3301
Mid (from experience)     855
Senior                    419
Entry/Junior              192
Director/Executive         92
Manager/Lead               90
Name: count, dtype: int64

## 4. Compare against the existing title-based `seniority` column

Two things to look at separately:

1. **Agreement rate** where both signals exist and the title-based tier wasn't the catch-all bucket — sanity-checks whether the two methods broadly agree when title *did* give a real signal.
2. **Resolution rate within `Mid/Unspecified`** — how many of the 2,843 ambiguous rows now get a concrete tier from the description text. This is the actual payoff of this notebook.

In [5]:
# 1. Agreement rate on non-ambiguous title tiers (collapse the experience-only "Mid" label to
#    compare fairly against the title column's "Mid/Unspecified")
comparable = df[df["seniority"] != "Mid/Unspecified"].copy()
comparable = comparable[comparable["seniority_from_experience"].notna()]

comparable["seniority_from_experience_norm"] = comparable["seniority_from_experience"].replace(
    {"Mid (from experience)": "Mid/Unspecified"}
)

agree = (comparable["seniority"] == comparable["seniority_from_experience_norm"]).mean()
print(f"Agreement rate where title gave a non-ambiguous tier AND description gave a year signal: {agree:.1%}")
print(f"(n = {len(comparable)})")

pd.crosstab(comparable["seniority"], comparable["seniority_from_experience"])

Agreement rate where title gave a non-ambiguous tier AND description gave a year signal: 26.1%
(n = 836)


seniority_from_experience,Director/Executive,Entry/Junior,Manager/Lead,Mid (from experience),Senior
seniority,,,,,
Director/Executive,43,0,25,5,23
Entry/Junior,0,29,0,25,0
Manager/Lead,36,9,47,159,150
Senior,6,15,7,158,99


In [6]:
# 2. Resolution rate within the Mid/Unspecified catch-all
unspecified = df[df["seniority"] == "Mid/Unspecified"]
resolved = unspecified["seniority_from_experience"].notna()

print(f"Mid/Unspecified rows: {len(unspecified)}")
print(f"Resolved by an experience-year mention: {resolved.sum()} ({resolved.mean():.1%})")
unspecified.loc[resolved, "seniority_from_experience"].value_counts()

Mid/Unspecified rows: 2843
Resolved by an experience-year mention: 812 (28.6%)


seniority_from_experience
Mid (from experience)    508
Senior                   147
Entry/Junior             139
Manager/Lead              11
Director/Executive         7
Name: count, dtype: int64

## 5. Build a reconciled `seniority_final` column

Rule: keep the title-based tier wherever it wasn't the catch-all bucket (it's a more direct signal than inferred years). Only fall back to the experience-derived tier for rows that were `Mid/Unspecified` **and** have a description year-signal. Rows that remain unresolved stay `Mid/Unspecified` — this notebook narrows the ambiguous bucket, it doesn't eliminate it.

In [7]:
def reconcile_final(row):
    if row["seniority"] != "Mid/Unspecified":
        return row["seniority"]
    if pd.notna(row["seniority_from_experience"]):
        return row["seniority_from_experience"]
    return "Mid/Unspecified"


df["seniority_final"] = df.apply(reconcile_final, axis=1)

before = df["seniority"].value_counts()
after = df["seniority_final"].value_counts()
pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)

,before,after
Director/Executive,219,226
Entry/Junior,248,387
Manager/Lead,967,978
Mid (from experience),0,508
Mid/Unspecified,2843,2031
Senior,672,819


## 6. Save enriched output

Ships `experience_years_required`, `seniority_from_experience`, and `seniority_final` alongside every original column. `seniority` (the original title-based column) is left untouched — this is an additive enrichment, not an overwrite, so the original Power BI build and prior findings in `Inferences.md` remain reproducible against the unmodified column.

In [8]:
df.drop(columns=["years_found"]).to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(f"Rows: {len(df)}")
print(f"Mid/Unspecified narrowed from {before.get('Mid/Unspecified', 0)} to {after.get('Mid/Unspecified', 0)}")

Saved: ../Outputs/mia_postings_seniority_enriched.csv
Rows: 4949
Mid/Unspecified narrowed from 2843 to 2031


## Limitations (carry into the report's limitations section if this feeds the write-up)

- **~33% coverage overall, 28.6% within `Mid/Unspecified` specifically.** Most postings don't state years-of-experience explicitly — this narrows the ambiguous bucket, it doesn't close it (2,843 → 2,031 rows remain unresolved).
- **Heuristic year-to-tier thresholds**, not a validated industry standard — stated explicitly above, should be stated explicitly in the report too if this column is cited.
- **Regex, not NLP** — will miss non-standard phrasings ("seasoned professional," "proven track record") and could mismatch on edge cases like "3 years of relevant coursework" being misread as work experience. Spot-check a sample before treating `seniority_final` as ground truth.
- **Median-of-lows reconciliation** is a defensible but not the only reasonable choice — max-of-highs or first-mention would each shift some borderline rows to a different tier.
- **Tested against the real dataset before this notebook was finalized** (not just against a sample): an earlier version of `years_to_tier` used `if years is None` instead of `pd.isna(years)`, which silently mislabeled all 2,031 unresolved rows as `Director/Executive` instead of leaving them `None`. Caught by checking the output distribution against expectations before trusting it — worth the same check if this logic is modified later.